# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NimaWyd/Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

> **[DRAFT — review and rewrite in your own words before submitting]**

**Task type: Scoring**

The Ranking Signal Analysis lane maps to a **scoring** task. The model assigns each content page a numerical priority score — higher score means higher estimated review urgency — and the editor team works through the pages in score order, top to bottom.

This is distinct from pure binary classification: the output of a classifier is "declining / not declining," which tells you nothing about *which* declining pages matter most. A score produces a ranked queue, which is exactly what a content team with a limited weekly review budget needs — they can draw a cut-line wherever their capacity allows.

It also differs from clustering (we have an observed label, so this is supervised) and from a full learning-to-rank setup (we don't have pairwise relevance judgements — we have a binary label per page that we use to score and then sort).

The one-paragraph frame:

> For a **content / SEO team**, deciding **which pages to prioritise for review this week**, we will build a **priority scorer** from 90-day search and engagement signals, predicting **likelihood of impression decline** (measured by `is_declining_label`) using **precision@K** as the primary metric. A wrong call (flagging pages that don't need work) costs wasted editor hours; missing a silent decline costs competitive ranking slots that are harder to recapture than to defend. A plain rule isn't enough because no single signal cleanly separates declining from non-declining pages — position tier, CTR, engagement, and freshness all carry partial information, and their interactions shift by client and content type.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Derive the target (prep step adds this column; we recreate it here for clarity)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

n = len(df)
n_declining = df["is_declining_label"].sum()
base_rate = n_declining / n

print(f"Dataset: {n:,} rows  |  {df['client_id'].nunique()} clients")
print(f"\nLabel distribution (is_declining_label):")
print(f"  Declining (1):     {n_declining:,}  ({base_rate:.1%})")
print(f"  Not declining (0): {n - n_declining:,}  ({1-base_rate:.1%})")
print(f"\nBase rate = {base_rate:.3f}  — a random scorer achieves this precision at any K.")

Dataset: 30,000 rows  |  32 clients

Label distribution (is_declining_label):
  Declining (1):     16,262  (54.2%)
  Not declining (0): 13,738  (45.8%)

Base rate = 0.542  — a random scorer achieves this precision at any K.


## 2. Target or proxy

> **[DRAFT — review and rewrite in your own words before submitting]**

**Target: `is_declining_label` — observed, not hand-defined**

The target column is `is_declining_label`: 1 if a page's impression count fell more than 20% from the earlier 30-day window (days 31–60 back) to the more recent 30-day window (days 1–30), else 0.

Crucially, the thing being measured — impression counts — is an **observed outcome** reported by Google Search Console. The 20% threshold is a rule applied to a real measurement; it is not a business opinion substituted for data. This satisfies the framing principle: the target must be observed, not defined.

**The label trap to avoid:** `trend_direction` (the string column from which the label is derived) and `trend_pct` (the underlying percentage) must **never** appear as model features. Including either would mean the model learns to re-derive the label from itself — a circular leak that inflates metrics on paper but produces a useless scorer in production.

**Why this target supports the scoring task:** A scorer trained to predict `is_declining_label` produces a probability per page. Sorting by that probability gives the prioritised review queue. Pages near the decision boundary (probability ~0.5–0.7) are the ones that benefit most from human review; high-confidence cases (>0.9) are clear triage; low-confidence cases (<0.3) can safely be deprioritised.

In [2]:
# Confirm the target column and the leakage guard
leakage_cols = ["trend_direction", "trend_pct"]
print("Label-source columns — must NEVER be model features:")
for col in leakage_cols:
    print(f"  {col!r}: {df[col].dtype}, {df[col].nunique()} unique values")

print()
print("Target column: is_declining_label")
print(df[["content_id", "trend_direction", "trend_pct", "is_declining_label"]]
      .head(8)
      .to_string(index=False))
print()
print("is_declining_label == (trend_direction == 'down')?",
      (df["is_declining_label"] == (df["trend_direction"] == "down").astype(int)).all())

Label-source columns — must NEVER be model features:
  'trend_direction': object, 5 unique values
  'trend_pct': float64, 2712 unique values

Target column: is_declining_label
          content_id trend_direction  trend_pct  is_declining_label
content_304f48230142            down      -41.4                   1
content_a1fb4e703a9e            down      -57.7                   1
content_9aa793d4d895            down      -60.9                   1
content_331d6c4de07b          stable      -13.8                   0
content_d99b7a2d90ca            down      -34.7                   1
content_d4084a4bc775            down      -38.9                   1
content_9a34b442b552            down      -92.3                   1
content_a63219c6e95a          stable        0.6                   0

is_declining_label == (trend_direction == 'down')? True


## 3. Success metric

> **[DRAFT — review and rewrite in your own words before submitting]**

**Primary metric: Precision@K**

For a scorer that produces a prioritised review queue, the question that matters is: *of the top K pages I hand to the editor team, how many actually needed attention?*

That is **precision@K** — and K is set by the team's weekly review capacity, not by the model.

**The baseline to beat:** Because 54.2% of pages in the dataset are declining, a random selection already achieves ~54.2% precision at any K. That is the floor. A scorer that doesn't materially exceed it adds no value over random triage.

**A realistic target:** If the scorer can push precision@K to, say, 70–75% at the team's working K (e.g. top 200 pages per week across clients), the team is spending roughly 30% fewer hours on pages that didn't need review — a meaningful, measurable improvement.

**Secondary check — Recall@K:** High precision with collapsing recall means the scorer is surfacing only the most obvious cases and missing a long tail of quietly declining pages. Both metrics together tell the honest story. Neither alone is sufficient.

**Why not ROC-AUC?** ROC-AUC measures ranking quality across all possible thresholds, which is useful during model development. But in deployment, the team operates at one threshold (their weekly K), so precision@K and recall@K at that operating point are more directly actionable than a curve.

**One number that means "good":** Precision@200 ≥ 0.70 at the operating K, compared to a 0.542 baseline.

In [3]:
# Establish the baseline precision and compare a naive heuristic
K = 200

# Baseline: random selection = base rate
base_rate = df["is_declining_label"].mean()
print(f"Base rate (random precision@any K): {base_rate:.3f}  ({base_rate:.1%})")

# Naive heuristic: sort by highest impressions (high-traffic pages first)
top_K_impressions = df.nlargest(K, "impressions_90d")["is_declining_label"].mean()
print(f"\nHeuristic — top {K} pages by impressions_90d:")
print(f"  Precision@{K}: {top_K_impressions:.3f}  ({top_K_impressions:.1%})")
print(f"  → This is BELOW the base rate — high-traffic pages are less likely to be declining.")
print(f"    Sorting by volume actually anti-selects for decline, the opposite of what we want.")
print()
print(f"This is why the metric and the scorer must be designed together:")
print(f"  A useful precision@{K} target: ≥ 0.70  (vs {base_rate:.3f} random baseline)")

Base rate (random precision@any K): 0.542  (54.2%)

Heuristic — top 200 pages by impressions_90d:
  Precision@200: 0.395  (39.5%)
  → This is BELOW the base rate — high-traffic pages are less likely to be declining.
    Sorting by volume actually anti-selects for decline, the opposite of what we want.

This is why the metric and the scorer must be designed together:
  A useful precision@200 target: ≥ 0.70  (vs 0.542 random baseline)


## 4. The unit of analysis, as a real dataframe

> **[DRAFT — review and rewrite in your own words before submitting]**

**One row = one pseudonymised content page**

The unit of analysis is a single content page: a URL managed by one client, with its 90-day aggregate search and engagement metrics attached. The grain is `content_id` — unique per row (confirmed in ML-02).

The scorer operates at this grain: it receives one row per page and returns one score per page. The score is then sorted within each client to produce that client's prioritised review queue.

The key columns for this task:

| Column | Role | Notes |
|---|---|---|
| `content_id` | Identifier | Grouping/joins only — never a feature |
| `client_id` | Group key | Used for client-holdout train/test splits |
| `content_type` | Feature candidate | Systematic missingness by type — add flag |
| `impression_tier` | Feature candidate | Volume bucket |
| `position_tier` | Feature candidate | GSC avg position bucket |
| `ctr` | Feature candidate | ×100 percentage (0.76 = 0.76%) |
| `engagement_rate` | Feature candidate | ×100 percentage |
| `word_count_tier` | Feature candidate | Blank for ~26% of rows — add `has_word_count` flag |
| `freshness_tier` | Feature candidate | Days-since-update bucket |
| `is_declining_label` | **Target** | 1 = declining; derived from `trend_direction` |

In [4]:
# Show the unit of analysis as a concrete dataframe slice
display_cols = [
    "content_id", "client_id", "content_type",
    "impression_tier", "position_tier",
    "ctr", "engagement_rate", "word_count_tier", "freshness_tier",
    "is_declining_label",
]

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Grain: content_id is unique = {df['content_id'].nunique() == len(df)}")
print(f"\nOne row = one pseudonymised content page. First 8 rows, key columns:\n")
print(df[display_cols].head(8).to_string(index=False))

print(f"\nTarget column summary:")
print(df["is_declining_label"].value_counts().rename({1: "declining (1)", 0: "not declining (0)"}).to_string())

Shape: 30,000 rows × 45 columns
Grain: content_id is unique = True

One row = one pseudonymised content page. First 8 rows, key columns:

          content_id         client_id    content_type impression_tier position_tier  ctr  engagement_rate word_count_tier freshness_tier  is_declining_label
content_304f48230142 client_f369cb89fc keyword article            good      striking 0.76             5.88       2000-3500           0-30                   1
content_a1fb4e703a9e client_4e07408562 keyword article            good      page_3_5 0.05             0.00       2000-3500           0-30                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            good      page_3_5 0.09             0.00           3500+           0-30                   1
content_331d6c4de07b client_19581e27de keyword article            good        page_1 0.49             1.28             NaN           0-30                   0
content_d99b7a2d90ca client_3fdba35f04 keyword article            good  

## 5. Why ML beats a fixed rule here

> **[DRAFT — review and rewrite in your own words before submitting]**

**The pattern is real but too messy for an if-statement.**

The obvious rule candidates each fail in a specific, instructive way:

- **"Flag all pages not on page 1"** misses a large fraction of the problem: 57% of page-1 pages are already declining. A rule that ignores position-1 pages would skip more than half the declining content.
- **"Flag all pages with below-median CTR"** is better, but 46–50% of high-CTR pages are also declining. Low CTR and high CTR both contain substantial decline signal — the threshold doesn't discriminate cleanly.
- **"Flag all pages that haven't been updated recently"** is correlated with decline, but freshness interacts with content type, keyword competition, and impression volume in ways a single threshold can't capture.

The core problem is **signal interaction**: a page with moderate CTR, good position, but no recent update, in a competitive keyword category, may be at much higher risk than a page that looks similar on any one dimension. The joint pattern is informative; the marginal signals, taken individually, are too noisy.

ML earns its place when:
1. Many signals carry *partial* information — too many to tune thresholds by hand.
2. The signals interact — risk in one dimension depends on the value in another.
3. The pattern shifts over time and across clients — hand-tuned rules go stale; a re-trained scorer adapts.

All three conditions hold here. The code below shows conditions 1 and 2 directly.

In [5]:
# Show that no single signal cleanly separates declining from non-declining pages

# Rule 1: "flag pages NOT on page_1" — what does it miss?
pg1 = df[df["position_tier"] == "page_1"]
non_pg1 = df[df["position_tier"] != "page_1"]
print("Rule 1: 'flag pages NOT on page_1'")
print(f"  Declining rate in page_1 pages (missed):       {pg1['is_declining_label'].mean():.1%}  "
      f"({pg1['is_declining_label'].sum():,} pages)")
print(f"  Declining rate in non-page_1 pages (flagged):  {non_pg1['is_declining_label'].mean():.1%}")
print(f"  → {pg1['is_declining_label'].sum():,} truly declining pages are IGNORED by this rule.\n")

# Rule 2: "flag pages with below-median CTR"
median_ctr = df["ctr"].median()
low_ctr  = df[df["ctr"] <  median_ctr]
high_ctr = df[df["ctr"] >= median_ctr]
print(f"Rule 2: 'flag pages with CTR below median ({median_ctr:.2f}%)'")
print(f"  Declining rate in low-CTR group (flagged):  {low_ctr['is_declining_label'].mean():.1%}")
print(f"  Declining rate in high-CTR group (ignored): {high_ctr['is_declining_label'].mean():.1%}  "
      f"({high_ctr['is_declining_label'].sum():,} declining pages missed)")
print()

# Signal interaction: within page_1, split by above/below-median CTR
pg1_median_ctr = pg1["ctr"].median()
pg1_low  = pg1[pg1["ctr"] <  pg1_median_ctr]
pg1_high = pg1[pg1["ctr"] >= pg1_median_ctr]
print(f"Signal interaction — page_1 pages split by CTR (median = {pg1_median_ctr:.2f}%):")
print(f"  page_1 + low CTR:  decline rate = {pg1_low['is_declining_label'].mean():.1%}  (n={len(pg1_low):,})")
print(f"  page_1 + high CTR: decline rate = {pg1_high['is_declining_label'].mean():.1%}  (n={len(pg1_high):,})")
print()
print("Conclusion: even within a single position tier, CTR carries additional signal.")
print("The joint pattern requires multiple signals — a single threshold rule is insufficient.")

Rule 1: 'flag pages NOT on page_1'
  Declining rate in page_1 pages (missed):       57.0%  (6,730 pages)
  Declining rate in non-page_1 pages (flagged):  52.4%
  → 6,730 truly declining pages are IGNORED by this rule.

Rule 2: 'flag pages with CTR below median (0.07%)'
  Declining rate in low-CTR group (flagged):  52.0%
  Declining rate in high-CTR group (ignored): 56.4%  (8,565 declining pages missed)

Signal interaction — page_1 pages split by CTR (median = 0.16%):
  page_1 + low CTR:  decline rate = 60.3%  (n=5,870)
  page_1 + high CTR: decline rate = 53.6%  (n=5,944)

Conclusion: even within a single position tier, CTR carries additional signal.
The joint pattern requires multiple signals — a single threshold rule is insufficient.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

---

**Note on this draft:** All markdown cells above are marked `[DRAFT]`. Before submitting, rewrite them in your own words — the framing decisions are yours to own. The code cells are intended to be run as-is; verify the numbers match the data before signing off.